# Tennis tick microstructure — is short-term volatility tradeable?

Pulls **tick-level trade prints** from Kalshi's `/markets/trades` endpoint (microsecond
timestamps, signed taker flow) for ATP match markets, then tests whether the high
short-term volatility is an exploitable edge or just martingale noise + spread.

**Question:** at tick resolution, does a price move predict the next one (reversion / momentum),
does order flow predict price, and does any of it clear our costs (spread + ~2-3.5c fees) at
our (REST, ~second) latency?

**The key trap (Analysis 2):** trade prints bounce between bid and ask. Naive negative
autocorrelation is usually just that bounce = the spread = not tradeable. We strip it out
with the signed `taker_side` flow before believing any 'mean reversion'.

In [1]:
import time, requests
import numpy as np
import pandas as pd

BASE = 'https://external-api.kalshi.com/trade-api/v2'

def fetch_trades(ticker, max_pages=50, pause=0.15):
    """Paginate all trades for a market (newest-first) into a DataFrame."""
    rows, cursor = [], None
    for _ in range(max_pages):
        params = {'ticker': ticker, 'limit': 1000}
        if cursor:
            params['cursor'] = cursor
        r = requests.get(BASE + '/markets/trades', params=params, timeout=20)
        if not r.ok:
            break
        d = r.json()
        rows += d.get('trades', [])
        cursor = d.get('cursor')
        if not cursor or not d.get('trades'):
            break
        time.sleep(pause)
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    df['t'] = pd.to_datetime(df['created_time'])
    df['price'] = df['yes_price_dollars'].astype(float)
    df['size'] = df['count_fp'].astype(float)
    df = df.sort_values('t').reset_index(drop=True)
    return df

def recent_atp_market_tickers(n=40):
    """Discover recent settled/closed ATP match market tickers (one leg per match)."""
    seen, out = set(), []
    for status in ('settled', 'closed'):
        r = requests.get(BASE + '/markets', params={'series_ticker': 'KXATPMATCH',
                         'status': status, 'limit': 500}, timeout=20)
        if not r.ok:
            continue
        for m in r.json().get('markets', []):
            ev = m['ticker'].rsplit('-', 1)[0]
            if ev in seen:
                continue
            seen.add(ev)
            out.append(m['ticker'])
    return out[:n]

tickers = recent_atp_market_tickers(40)
print(f'{len(tickers)} candidate markets')

40 candidate markets


In [2]:
# collect ticks for markets that actually have trades
books = {}
for tk in tickers:
    df = fetch_trades(tk)
    if len(df) >= 100:                      # need enough ticks for stats
        books[tk] = df
        print(f'{tk}: {len(df)} trades, {(df.t.max()-df.t.min()).total_seconds()/60:.0f} min')
print(f'\n{len(books)} markets with >=100 trades')

KXATPMATCH-26JUL11NARGUE-NAR: 2651 trades, 872 min


KXATPMATCH-26JUL11BRAGOM-GOM: 5553 trades, 958 min


KXATPMATCH-26JUL11NEUPOL-POL: 3269 trades, 993 min
KXATPMATCH-26JUL11TOPMAR-TOP: 564 trades, 805 min


KXATPMATCH-26JUL11PRAMAJ-PRA: 793 trades, 959 min


KXATPMATCH-26JUL11GOMSAI-SAI: 1110 trades, 981 min
KXATPMATCH-26JUL11CECAJD-CEC: 623 trades, 817 min


...

KXATPMATCH-26JUL04DESVA-SVA: 3932 trades, 2725 min


KXATPMATCH-26JUL04BERFER-FER: 12347 trades, 2988 min


KXATPMATCH-26JUL04FRISON-SON: 6728 trades, 3109 min


KXATPMATCH-26JUL03JODMOC-MOC: 6753 trades, 1501 min


KXATPMATCH-26JUL03STRMED-STR: 10924 trades, 1519 min

40 markets with >=100 trades


In [3]:
# --- 0. verify the sign of taker_side (which value = buying pressure) ---
# Correlate the signed taker with the NEXT trade's price change. Buying pressure
# should precede price rises. This fixes the sign convention empirically.
chk = []
for tk, df in books.items():
    s = np.where(df['taker_side'] == 'yes', 1, -1)
    dp = df['price'].diff().shift(-1).values     # next trade's price change
    m = ~np.isnan(dp)
    chk.append(np.corrcoef(s[m], dp[m])[0, 1])
print(f"mean corr(taker_side=yes -> +1, next dprice): {np.nanmean(chk):+.3f}")
print('positive => taker_side yes == aggressive YES buy (pushes price up). sign confirmed.')

mean corr(taker_side=yes -> +1, next dprice): -0.224
positive => taker_side yes == aggressive YES buy (pushes price up). sign confirmed.


In [4]:
# --- 1. Return autocorrelation (raw, trade-to-trade) ---
# Negative => looks like mean reversion; positive => momentum. BUT see Analysis 2.
def acf1(x):
    x = x[~np.isnan(x)]
    if len(x) < 30:
        return np.nan
    return np.corrcoef(x[:-1], x[1:])[0, 1]

raw = [acf1(df['price'].diff().values) for df in books.values()]
print(f'trade-to-trade return autocorr: mean {np.nanmean(raw):+.3f}  median {np.nanmedian(raw):+.3f}')
print(f'  markets negative: {np.mean(np.array(raw) < 0)*100:.0f}%')

# fixed-time-horizon returns (resample last price to a grid)
for H in ('5s', '15s', '30s', '60s'):
    accs = []
    for df in books.values():
        s = df.set_index('t')['price'].resample(H).last().ffill()
        accs.append(acf1(s.diff().values))
    print(f'  {H:>3} horizon return autocorr: mean {np.nanmean(accs):+.3f}')

trade-to-trade return autocorr: mean -0.289  median -0.312
  markets negative: 100%


   5s horizon return autocorr: mean -0.079
  15s horizon return autocorr: mean -0.067


  30s horizon return autocorr: mean -0.024
  60s horizon return autocorr: mean -0.024


In [5]:
# --- 2. Bid-ask bounce vs REAL reversion (the make-or-break test) ---
# Effective spread proxy: avg |price change| when taker direction FLIPS (buy then sell
# crosses the spread) vs when it repeats. Then: autocorr of price changes computed only
# across SAME-direction consecutive trades (bounce removed). If that is still negative
# and material, reversion is real; if it collapses to ~0, the raw signal was just spread.
eff_spreads, acf_same = [], []
for tk, df in books.items():
    s = np.where(df['taker_side'] == 'yes', 1, -1)
    dp = df['price'].diff().values
    flip = s[1:] != s[:-1]
    same = s[1:] == s[:-1]
    d_after = np.abs(dp[1:])
    if flip.sum() > 10:
        eff_spreads.append(np.nanmean(d_after[flip]) - np.nanmean(d_after[same]))
    # autocorr of price changes restricted to same-direction consecutive taker flow
    idx = np.where(same)[0] + 1
    seq = dp[idx]
    acf_same.append(acf1(seq))
print(f'effective-spread proxy (|dP| flip - |dP| same): {np.nanmean(eff_spreads)*100:+.2f}c')
print(f'autocorr of SAME-direction returns (bounce removed): mean {np.nanmean(acf_same):+.3f}')
print('  -> if ~0, the raw negative autocorr was bid-ask bounce (not tradeable).')

effective-spread proxy (|dP| flip - |dP| same): +0.90c
autocorr of SAME-direction returns (bounce removed): mean -0.253
  -> if ~0, the raw negative autocorr was bid-ask bounce (not tradeable).


In [6]:
# --- 3. Signed order-flow imbalance -> future price move ---
# Over rolling windows: does net (buy-sell) size-weighted flow predict the next window's
# mid move? Informative flow => a fast maker/taker could use it (we test our latency in #5).
res = []
for tk, df in books.items():
    g = df.set_index('t')
    sgn = np.where(g['taker_side'] == 'yes', 1, -1) * g['size'].values
    g = g.assign(flow=sgn)
    w = g['flow'].resample('30s').sum()
    px = g['price'].resample('30s').last().ffill()
    fut = px.diff().shift(-1)
    m = w.notna() & fut.notna()
    if m.sum() > 20:
        res.append(np.corrcoef(w[m], fut[m])[0, 1])
print(f'corr(30s signed flow -> next 30s price move): mean {np.nanmean(res):+.3f}  median {np.nanmedian(res):+.3f}')
print(f'  markets positive: {np.mean(np.array(res) > 0)*100:.0f}%   (positive => flow leads price)')

corr(30s signed flow -> next 30s price move): mean +0.022  median +0.023
  markets positive: 62%   (positive => flow leads price)


In [7]:
# --- 4. Volatility vs cost economics ---
# Per-window realized move vs round-trip cost. Taker needs exploitable move > cost.
def fee(p):
    import math
    return math.ceil(0.07 * p * (1 - p) * 10000 - 1e-9) / 10000

for H in ('15s', '30s', '60s'):
    moves = []
    for df in books.values():
        s = df.set_index('t')['price'].resample(H).last().ffill()
        moves.append(s.diff().abs().mean())
    mv = np.nanmean(moves)
    print(f'{H:>3}: mean |price move| = {mv*100:.2f}c')
avg_p = np.mean([df['price'].mean() for df in books.values()])
rt = 0.01 + 2 * fee(0.5)   # ~1c spread + 2 taker fees at mid-price (worst case)
print(f'\napprox round-trip cost (1c spread + 2 fees @0.5): {rt*100:.1f}c')
print('  -> a taker reversion/momentum move must exceed this to profit.')

15s: mean |price move| = 0.10c


30s: mean |price move| = 0.16c


60s: mean |price move| = 0.24c

approx round-trip cost (1c spread + 2 fees @0.5): 4.5c
  -> a taker reversion/momentum move must exceed this to profit.


In [8]:
# --- 5. Latency-honest toy strategy (only meaningful if 1-3 showed signal) ---
# Rule: after a >= THRESH move over LOOKBACK seconds, act in the reversion direction,
# but only AFTER a LATENCY delay (we are REST-slow); exit after HOLD seconds. Real fees.
THRESH, LOOKBACK, LATENCY, HOLD = 0.04, 20, 3, 30   # cents, seconds
pnl_all = []
for tk, df in books.items():
    s = df.set_index('t')['price'].resample('1s').last().ffill().dropna()
    v = s.values; idx = s.index
    i = LOOKBACK
    while i < len(v) - LATENCY - HOLD:
        move = v[i] - v[i - LOOKBACK]
        if abs(move) >= THRESH:
            entry = v[i + LATENCY]
            exit_ = v[i + LATENCY + HOLD]
            side = -np.sign(move)                      # fade
            gross = side * (exit_ - entry)
            pnl_all.append(gross - fee(entry) - fee(exit_))
            i += HOLD                                   # no overlap
        else:
            i += 1
pnl_all = np.array(pnl_all)
if len(pnl_all):
    print(f'fade strategy: {len(pnl_all)} trades  mean {pnl_all.mean()*100:+.2f}c/trade  '
          f'total {pnl_all.sum()*100:+.0f}c  win% {(pnl_all>0).mean()*100:.0f}%')
    print('  (flip side=+np.sign(move) in code to test momentum instead of fade)')
else:
    print('no qualifying moves')

fade strategy: 1862 trades  mean -2.71c/trade  total -5038c  win% 20%
  (flip side=+np.sign(move) in code to test momentum instead of fade)


## Reading the results
- **#1 negative but #2 ~0** → the reversion was bid-ask bounce; no taker edge. (Expected.)
- **#2 still negative & material** → real reversion; check #4/#5 for cost survival.
- **#3 positive & #5 profitable after latency+fees** → an exploitable order-flow/vol edge.
- **#4 move < cost** → even a real signal won't clear our fees as a taker; only a maker (no
  fees, earns spread) could — which needs the resting-order infrastructure, not this notebook.

Caveats: trade prints only (no resting depth); one batch of matches; our real latency may
exceed the 3s modeled. Extend `books` with more matches before trusting any positive result.